# Colab GPU 딥러닝 실습: MNIST 손글씨 숫자 분류

과제 3번: **GPU 연결 및 ML/DL 실행 결과 스크린샷 5장 (시각화 포함)**

이 노트북은 공개 MNIST 데이터로 합성곱 신경망(CNN)을 학습합니다. 정비 DB는 사용하거나 업로드하지 않습니다.

## 실행 순서
1. Colab의 **런타임 → 런타임 유형 변경 → 사용 가능한 GPU (예: T4) → 저장**을 선택합니다.
2. 첫 코드 셀의 STUDENT_NAME을 본인 이름으로 바꿉니다.
3. 위에서부터 코드 셀을 차례로 실행합니다. 첫 셀에서 GPU가 없으면 중단합니다.
4. 각 **증빙 1~5** 제목과 실행 결과가 함께 보이도록 실제 Colab 화면을 캡처합니다.
5. 완료 후 **파일 → 다운로드 → .ipynb 다운로드**로 실행 결과가 들어 있는 노트북도 보관합니다.

아직 실행 결과가 없는 원본 노트북입니다. 정확도·GPU 종류·학습시간은 실행 환경에 따라 달라집니다. 로컬에서 만든 그래프를 Colab 실행 화면으로 대신하지 마세요.

## 학습 목표
- CPU와 GPU의 차이 및 TensorFlow GPU 장치 확인
- 이미지 정규화, 훈련/검증/시험 데이터 분리
- CNN 학습, 검증 정확도와 손실 변화 분석
- 시험 정확도, 혼동행렬 및 오분류 사례 확인

참고: [TensorFlow MNIST 예제](https://www.tensorflow.org/tutorials/quickstart/beginner), [Colab 안내](https://research.google.com/colaboratory/faq.html)


## 증빙 1 / 5 — GPU 연결 및 실제 GPU 연산

아래 셀에서 GPU 이름, TensorFlow GPU 목록, GPU 행렬 연산 장치가 표시됩니다. **GPU가 없으면 다음 실습으로 진행하지 말고 런타임 설정을 변경하세요.**

캡처 파일명: **01_Colab_GPU_연결.png**


In [ ]:
STUDENT_NAME = "본인 이름으로 변경"

import os, sys, json, time, uuid, subprocess, platform
from pathlib import Path
from datetime import datetime, timezone, timedelta
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
tf.keras.utils.set_random_seed(SEED)
RUN_ID = datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
OUTPUT_DIR = Path("/content/mnist_gpu_results") / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gpus = tf.config.list_physical_devices("GPU")
assert gpus, "GPU가 없습니다. 런타임 → 런타임 유형 변경 → GPU 선택 후 첫 셀부터 다시 실행하세요."
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    text=True,
).strip()
with tf.device("/GPU:0"):
    gpu_check = tf.matmul(tf.ones((256, 256)), tf.ones((256, 256)))
    gpu_sum = float(tf.reduce_sum(gpu_check).numpy())
assert "GPU" in gpu_check.device, "행렬 연산이 GPU에 배치되지 않았습니다."

def evidence_header(number, title):
    print("=" * 72)
    print(f"EVIDENCE {number}/5 | {title}")
    print(f"Student: {STUDENT_NAME} | Run: {RUN_ID}")
    print(f"GPU: {gpu_info}")
    print("=" * 72)

evidence_header(1, "COLAB GPU CONNECTION")
print("Python:", platform.python_version(), "| TensorFlow:", tf.__version__)
print("TensorFlow GPU devices:", gpus)
print("GPU matrix result device:", gpu_check.device)
print("GPU matrix checksum:", gpu_sum)
print("GPU connection and actual matrix computation: SUCCESS")
if STUDENT_NAME == "본인 이름으로 변경":
    print("주의: 첫 줄 STUDENT_NAME을 본인 이름으로 수정하고 다시 실행하세요.")


## 증빙 2 / 5 — 데이터 준비 및 이미지 시각화

MNIST는 0~9 손글씨의 28×28 흑백 이미지입니다. 훈련용 60,000장 중 54,000장은 학습, 6,000장은 검증에 사용하며, 별도의 시험용 10,000장으로 마지막 성능을 평가합니다. 픽셀을 0~1로 정규화합니다.

캡처 파일명: **02_MNIST_데이터_시각화.png**


In [ ]:
(x_train_all, y_train_all), (x_test_raw, y_test) = tf.keras.datasets.mnist.load_data()

x_train_all = x_train_all.astype("float32")[..., np.newaxis] / 255.0
x_test = x_test_raw.astype("float32")[..., np.newaxis] / 255.0
order = np.random.default_rng(SEED).permutation(len(x_train_all))
x_train = x_train_all[order[:54000]]
y_train = y_train_all[order[:54000]]
x_val = x_train_all[order[54000:]]
y_val = y_train_all[order[54000:]]

evidence_header(2, "MNIST DATA AND VISUALIZATION")
print(f"Train: {x_train.shape} | Validation: {x_val.shape} | Test: {x_test.shape}")
print(f"Pixel range: [{x_train.min():.1f}, {x_train.max():.1f}] | Classes: 0-9")
fig, axes = plt.subplots(3, 6, figsize=(11, 4.8))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i, :, :, 0], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}", fontsize=10)
    ax.axis("off")
fig.suptitle(f"MNIST Training Samples | Run {RUN_ID}", fontsize=13)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_mnist_samples.png", dpi=160, bbox_inches="tight")
plt.show()


## 증빙 3 / 5 — CNN 모델의 GPU 학습

합성곱층으로 이미지 특징을 추출하고, 마지막 Dense 층의 softmax로 10개 숫자의 확률을 출력합니다. Adam 최적화와 교차엔트로피 손실을 사용합니다. 검증 데이터는 학습 가중치 업데이트에 사용하지 않습니다.

캡처 파일명: **03_CNN_GPU_학습.png**


In [ ]:
EPOCHS = 3
BATCH_SIZE = 128
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

with tf.device("/GPU:0"):
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    evidence_header(3, "CNN TRAINING ON GPU")
    print("Architecture: Conv2D(32) -> Pool -> Conv2D(64) -> Pool -> Dense(64) -> Dropout -> Dense(10)")
    print(f"Parameters: {model.count_params():,} | Epochs: {EPOCHS} | Batch size: {BATCH_SIZE}")
    print("Requested training device: /GPU:0")
    print("First weight device:", getattr(getattr(model.trainable_variables[0], "handle", None), "device", "See GPU tensor check in evidence 1"))
    start = time.perf_counter()
    history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                        epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)
    training_seconds = time.perf_counter() - start

print(f"Training completed in {training_seconds:.2f} seconds")
print(f"Final training accuracy: {history.history['accuracy'][-1]:.4%}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4%}")


## 증빙 4 / 5 — 시험 성능과 학습 곡선 시각화

학습·검증 정확도 및 손실을 비교합니다. 최종 시험 정확도는 학습과 검증에 쓰지 않은 시험 데이터에서 계산합니다. 그래프가 실제로 개선되는지, 훈련과 검증 사이에 차이가 커지는지 관찰하세요.

캡처 파일명: **04_정확도_손실_그래프.png**


In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
evidence_header(4, "TEST METRICS AND LEARNING CURVES")
print(f"Test samples: {len(y_test):,} | Test accuracy: {test_accuracy:.4%} | Test loss: {test_loss:.6f}")
epochs_axis = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs_axis, history.history["accuracy"], "o-", label="Train")
axes[0].plot(epochs_axis, history.history["val_accuracy"], "o-", label="Validation")
axes[0].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy", xticks=epochs_axis)
axes[1].plot(epochs_axis, history.history["loss"], "o-", label="Train")
axes[1].plot(epochs_axis, history.history["val_loss"], "o-", label="Validation")
axes[1].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy", xticks=epochs_axis)
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.25)
fig.suptitle(f"CNN Learning Curves | Test accuracy {test_accuracy:.2%} | Run {RUN_ID}", fontsize=11)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_learning_curves.png", dpi=160, bbox_inches="tight")
plt.show()


## 증빙 5 / 5 — 혼동행렬과 예측·오분류 사례

혼동행렬의 행은 실제 정답, 열은 모델 예측입니다. 오른쪽에는 정분류 예시와 오분류 예시를 최대 3개씩 표시합니다. 초록 제목은 정분류, 빨강 제목은 오분류입니다. **오분류가 나오는 것도 정상적인 실습 결과입니다.**

캡처 파일명: **05_혼동행렬_예측결과.png**


In [ ]:
probabilities = model.predict(x_test, batch_size=256, verbose=0)
predicted = np.argmax(probabilities, axis=1)
cm = np.zeros((10, 10), dtype=np.int64)
np.add.at(cm, (y_test, predicted), 1)
correct = np.flatnonzero(predicted == y_test)
wrong = np.flatnonzero(predicted != y_test)
selected = list(correct[:3]) + list(wrong[:3])
if len(selected) < 6:
    selected += [int(i) for i in correct if i not in selected][:6-len(selected)]

evidence_header(5, "CONFUSION MATRIX AND PREDICTIONS")
print(f"Correct: {len(correct):,} | Incorrect: {len(wrong):,} | Accuracy: {len(correct)/len(y_test):.4%}")
fig = plt.figure(figsize=(12, 5.2), layout="constrained")
grid = fig.add_gridspec(2, 5, width_ratios=[1.3, 1.3, 1, 1, 1])
ax_cm = fig.add_subplot(grid[:, :2])
ax_cm.imshow(cm, cmap="Blues")
ax_cm.set(xticks=range(10), yticks=range(10), xlabel="Predicted digit", ylabel="True digit", title="Confusion Matrix")
threshold = cm.max() * 0.5
for r in range(10):
    for c in range(10):
        ax_cm.text(c, r, str(cm[r, c]), ha="center", va="center", fontsize=7,
                   color="white" if cm[r, c] > threshold else "black")
for n, i in enumerate(selected):
    ax = fig.add_subplot(grid[n//3, 2+n%3])
    ax.imshow(x_test_raw[i], cmap="gray")
    ax.set_title(f"True {y_test[i]} / Pred {predicted[i]}\nConfidence {probabilities[i, predicted[i]]:.1%}",
                 fontsize=9, color="seagreen" if y_test[i] == predicted[i] else "firebrick")
    ax.axis("off")
fig.suptitle(f"CNN Test Predictions | Run {RUN_ID}", fontsize=12)
fig.savefig(OUTPUT_DIR / "05_confusion_and_predictions.png", dpi=160, bbox_inches="tight")
plt.show()


## 실행 결과 저장 (보조 자료)

아래 셀은 실제 실행한 수치, 학습 이력, 그림을 ZIP으로 저장합니다. 다운로드한 **그래프 이미지 자체는 Colab 화면 스크린샷을 대체하지 않습니다.** 과제 증빙 5장은 위 셀들의 실제 실행 화면을 직접 캡처하세요.

다운로드하려면 DOWNLOAD_RESULTS를 True로 바꾸고 이 셀을 실행합니다. 노트북도 실행 결과를 포함해 별도로 다운로드해 두세요.


In [ ]:
DOWNLOAD_RESULTS = False
import csv, shutil
metrics = {
    "student": STUDENT_NAME, "run_id": RUN_ID, "gpu": gpu_info,
    "tensorflow_version": tf.__version__, "gpu_check_device": gpu_check.device,
    "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "training_samples": len(y_train), "validation_samples": len(y_val), "test_samples": len(y_test),
    "training_seconds": training_seconds, "test_accuracy": float(test_accuracy), "test_loss": float(test_loss),
    "incorrect_test_predictions": int(len(wrong)),
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
with (OUTPUT_DIR / "training_history.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "accuracy", "val_accuracy", "loss", "val_loss"])
    for i in range(EPOCHS):
        writer.writerow([i+1] + [history.history[k][i] for k in ["accuracy", "val_accuracy", "loss", "val_loss"]])
model.save(OUTPUT_DIR / "mnist_cnn.keras")
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Actual execution results saved:", archive)
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive)


## 제출 전에 본인이 설명할 내용

- GPU 이름과 실제 GPU 연산 장치가 무엇이었는가?
- 이미지 픽셀을 0~1로 정규화한 이유는 무엇인가?
- 훈련·검증·시험 데이터는 각각 무엇에 사용했는가?
- 학습 곡선에서 정확도/손실은 어떻게 변했는가?
- 시험 정확도와 대표적인 오분류는 무엇이었는가?

정확도와 학습시간은 본인의 실제 출력값을 사용하세요. 다른 사람의 결과나 예상값을 자신의 실험 결과로 적지 마세요.

이 실습은 MNIST 분류 실습이며 MAI SMA 앱의 Llama 모델을 학습한 것은 아닙니다.
